# Assignment 2: Track Machine Learning Experiments Using MLflow
**Course:** MLOps  
**Assignment Title:** Track Machine Learning Experiments Using MLflow  
**Difficulty Level:** Intermediate  
**Programming Language:** Python  

---

## 1. Aim
To develop a Machine Learning classification model and use **MLflow** to track experiments by logging hyperparameters, evaluation metrics, visual artifacts, and trained models for reproducibility and comparison.

## 2. Problem Statement
Machine Learning development involves numerous experiments with different algorithms and hyperparameters. Managing these experiments manually is time-consuming and error-prone. This notebook implements an end-to-end MLOps workflow using Python and MLflow to record every experiment run, compare performance metrics, identify the best model, and register it to the MLflow Model Registry.

## 3. Workflow Steps
- **Step 1:** Load Dataset (`data/iris.csv`)
- **Step 2:** Preprocessing (missing value imputation, label encoding, standard feature scaling)
- **Step 3:** Split Data (80% Train, 20% Test)
- **Step 4:** Train at least three different models (Logistic Regression, Decision Tree, Random Forest)
- **Step 5:** Track every experiment using MLflow (parameters, accuracy, precision, recall, f1-score, confusion matrix, classification report, trained model)
- **Step 6:** Compare all experiments
- **Step 7:** Select the best model
- **Step 8:** Register the model in MLflow Model Registry

In [ ]:
# Step 0: Software & Library Imports
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Ensure compatibility and configure SQLite Tracking Backend
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
DB_PATH = os.path.abspath('../mlflow.db').replace('\\', '/')
mlflow.set_tracking_uri(f'sqlite:///{DB_PATH}')
EXPERIMENT_NAME = 'Iris_Classification_Experiment'
mlflow.set_experiment(EXPERIMENT_NAME)

print('MLflow Tracking URI:', mlflow.get_tracking_uri())
print('Active MLflow Experiment:', EXPERIMENT_NAME)

## Step 1: Load Dataset
We load the Iris dataset from `data/iris.csv` and inspect its shape, columns, and head.

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'iris.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset Shape: {df.shape}')
df.head()

## Step 2: Perform Preprocessing
- Check and handle missing values
- Apply `LabelEncoder` to encode target species
- Apply `StandardScaler` to scale numerical features

In [ ]:
# 1. Handle missing values
print('Missing values count:')
print(df.isnull().sum())

X_raw = df.drop(columns=['species'])
y_raw = df['species']

# 2. Label encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
target_names = list(label_encoder.classes_)
print('Target Encoding:', dict(enumerate(target_names)))

# 3. Feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
feature_names = list(X_raw.columns)
print('Preprocessed feature matrix shape:', X.shape)

## Step 3: Split Data (80% Train, 20% Test)
Using stratified train/test split to preserve class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set size: {X_train.shape[0]} samples (80%)')
print(f'Testing set size:  {X_test.shape[0]} samples (20%)')

## Steps 4 & 5: Train Models & Track Experiments with MLflow
We train three models:
1. **Logistic Regression**
2. **Decision Tree**
3. **Random Forest**

For each model run, MLflow logs:
- **Parameters:** model name, hyperparameters (`C`, `max_iter`, `criterion`, `n_estimators`, etc.)
- **Metrics:** Accuracy, Precision, Recall, F1 Score
- **Artifacts:** Confusion Matrix heatmap (PNG) and Classification Report (TXT & JSON)
- **Trained Model:** Serialized model logged into MLflow model store

In [ ]:
temp_dir = os.path.join('..', 'temp_artifacts')
os.makedirs(temp_dir, exist_ok=True)

models = [
    {
        'name': 'Logistic Regression',
        'run_name': 'NB_Logistic_Regression',
        'estimator': LogisticRegression(C=1.0, max_iter=200, solver='lbfgs', random_state=42),
        'params': {'C': 1.0, 'max_iter': 200, 'solver': 'lbfgs', 'random_state': 42}
    },
    {
        'name': 'Decision Tree',
        'run_name': 'NB_Decision_Tree',
        'estimator': DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_split=2, random_state=42),
        'params': {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 2, 'random_state': 42}
    },
    {
        'name': 'Random Forest',
        'run_name': 'NB_Random_Forest',
        'estimator': RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=2, random_state=42),
        'params': {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 2, 'random_state': 42}
    }
]

for entry in models:
    m_name = entry['name']
    r_name = entry['run_name']
    clf = entry['estimator']
    params = entry['params']
    
    with mlflow.start_run(run_name=r_name) as run:
        # Fit
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        print(f'[{m_name}] Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
        
        # Log parameters & metrics
        mlflow.log_param('model_name', m_name)
        for k, v in params.items():
            mlflow.log_param(k, v)
        
        mlflow.log_metric('Accuracy', acc)
        mlflow.log_metric('Precision', prec)
        mlflow.log_metric('Recall', rec)
        mlflow.log_metric('F1 Score', f1)
        mlflow.log_metric('f1_score', f1)
        mlflow.log_metric('accuracy', acc)
        
        # Artifact 1: Confusion Matrix Plot
        cm = confusion_matrix(y_test, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=ax)
        ax.set_title(f'Confusion Matrix: {m_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        plt.tight_layout()
        cm_file = os.path.join(temp_dir, f"{m_name.lower().replace(' ', '_')}_cm.png")
        fig.savefig(cm_file, dpi=200)
        plt.close(fig)
        mlflow.log_artifact(cm_file, artifact_path='plots')
        
        # Artifact 2: Classification Report
        rep_file = os.path.join(temp_dir, f"{m_name.lower().replace(' ', '_')}_report.txt")
        with open(rep_file, 'w') as f:
            f.write(classification_report(y_test, y_pred, target_names=target_names))
        mlflow.log_artifact(rep_file, artifact_path='reports')
        
        # Log Model
        mlflow.sklearn.log_model(
            sk_model=clf,
            artifact_path='model',
            pip_requirements=['scikit-learn>=1.3.0', 'numpy>=1.24.0', 'pandas>=2.0.0']
        )
print('All models trained and tracked in MLflow successfully!')

## Step 6: Compare All Experiments
Query the MLflow experiment runs and summarize their metrics in a clean comparison table.

In [ ]:
client = MlflowClient()
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=['metrics.F1 Score DESC'])

comparison_data = []
for r in runs:
    comparison_data.append({
        'Run Name': r.data.tags.get('mlflow.runName', 'N/A'),
        'Model Name': r.data.params.get('model_name', 'N/A'),
        'Accuracy': round(r.data.metrics.get('Accuracy', 0.0), 4),
        'Precision': round(r.data.metrics.get('Precision', 0.0), 4),
        'Recall': round(r.data.metrics.get('Recall', 0.0), 4),
        'F1 Score': round(r.data.metrics.get('F1 Score', 0.0), 4),
        'Run ID': r.info.run_id
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Metric Comparison Plot
plt.figure(figsize=(10, 5))
plot_df = comparison_df.drop_duplicates(subset=['Model Name']).copy()
x = np.arange(len(plot_df))
w = 0.18
plt.bar(x - 1.5*w, plot_df['Accuracy'], width=w, label='Accuracy', color='#1f77b4')
plt.bar(x - 0.5*w, plot_df['Precision'], width=w, label='Precision', color='#2ca02c')
plt.bar(x + 0.5*w, plot_df['Recall'], width=w, label='Recall', color='#ff7f0e')
plt.bar(x + 1.5*w, plot_df['F1 Score'], width=w, label='F1 Score', color='#d62728')
plt.xticks(x, plot_df['Model Name'])
plt.ylim(0.8, 1.05)
plt.ylabel('Metric Value')
plt.title('MLflow Experiment Model Comparison')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## Step 7: Select Best Model
Identify the model with the top F1 Score and Accuracy.

In [ ]:
best_run = comparison_df.iloc[0]
best_run_id = best_run['Run ID']
best_model_name = best_run['Model Name']

print('Best Model Identified:')
print(f'  Model:     {best_model_name}')
print(f'  Run ID:    {best_run_id}')
print(f'  Accuracy:  {best_run["Accuracy"]}')
print(f'  F1 Score:  {best_run["F1 Score"]}')

## Step 8: Register Best Model in MLflow Model Registry
Register the top-performing model version to enable versioning and deployment tracking.

In [ ]:
REGISTERED_MODEL_NAME = 'Iris_Best_Classifier'
model_uri = f'runs:/{best_run_id}/model'
registered_model = mlflow.register_model(model_uri=model_uri, name=REGISTERED_MODEL_NAME)

print(f'Model registered successfully as: {registered_model.name} (Version {registered_model.version})')

# Reload registered model and evaluate inference
loaded_model = mlflow.sklearn.load_model(f'models:/{REGISTERED_MODEL_NAME}/{registered_model.version}')
predictions = loaded_model.predict(X_test[:5])
print('\nInference test on 5 samples:')
for i, pred_idx in enumerate(predictions):
    print(f'Sample {i+1} -> Predicted: {target_names[pred_idx]} | True: {target_names[y_test[i]]}')

## MLflow UI Access Instructions
To view experiments, runs, artifacts, and registered models interactively in the web UI, open a terminal in this directory and run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then visit `http://127.0.0.1:5000` in your web browser.